# ใบงานที่ 3: Regression & Classification
### Age & Gender Prediction จากภาพใบหน้า (UTKFace Dataset)

**รายวิชา:** Machine Learning (04-624-201)
**มหาวิทยาลัยเทคโนโลยีราชมงคลธัญบุรี**

**ชื่อ:** ...........พีรภัทร...........พุดพันธ์............................  **รหัสนักศึกษา:** ........116710400685-9.....................

---

## ภาพรวมของงาน

โน้ตบุ๊กนี้ครอบคลุมเนื้อหาครบทั้ง 3 LAB ตามที่ใบงานกำหนด:

- **LAB 1: Regression** — ทำนายอายุ (Age) จากภาพใบหน้า ด้วย Simple และ Multiple Linear Regression
- **LAB 2: Classification** — ทำนายเพศ (Gender) จากภาพใบหน้า ด้วย Logistic Regression
- **LAB 3: Model Comparison** — เปรียบเทียบและวิเคราะห์ผลลัพธ์ทั้งหมด

**Dataset:** UTKFace (สุ่มมา 1,500 ภาพจากทั้งหมด 20,000+ ภาพ) — แต่ละภาพมี label อายุและเพศฝังอยู่ในชื่อไฟล์

**Feature Engineering:** ภาพถูก resize เป็น 64x64 grayscale → flatten เป็นเวกเตอร์ 4,096 มิติ → ลดมิติด้วย PCA เหลือ 50 components

## 0. Setup: Import Libraries

In [ ]:
import os
import random
import shutil
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
)

import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams["figure.dpi"] = 100
print("Import libraries สำเร็จ")

## 1. การเตรียมข้อมูล (Data Preparation)

ขั้นตอนนี้เป็นการเตรียมข้อมูลก่อนเข้าสู่ LAB 1-3 ประกอบด้วย:
1. สุ่มภาพจาก UTKFace มา 1,500 ภาพ
2. Parse label (age, gender) จากชื่อไฟล์
3. แปลงภาพเป็นตัวเลข (feature extraction) และลดมิติด้วย PCA

> **หมายเหตุ:** ถ้าเคยรันขั้นตอนนี้ไปแล้วและมีไฟล์ `data/features_pca.csv` อยู่แล้ว สามารถข้ามไปหัวข้อ 1.4 (โหลดผลลัพธ์) ได้เลย เพื่อประหยัดเวลา

### 1.1 สุ่มภาพจาก UTKFace (ข้ามได้ถ้าทำแล้ว)

Dataset ต้นฉบับ UTKFace ดาวน์โหลดจาก Kaggle: `jangedoo/utkface-new` แล้วแตกไฟล์ไว้ที่ `data/data/UTKFace_raw/UTKFace`

In [ ]:
SOURCE_DIR = "data/data/UTKFace_raw/UTKFace"
TARGET_DIR = "data/UTKFace_sampled"
N_SAMPLES = 1500

if not os.path.exists(TARGET_DIR) or len(os.listdir(TARGET_DIR)) < N_SAMPLES:
    os.makedirs(TARGET_DIR, exist_ok=True)
    all_files = [f for f in os.listdir(SOURCE_DIR) if f.endswith(".jpg")]
    print(f"ภาพทั้งหมดใน source: {len(all_files)}")

    random.seed(42)
    sampled_files = random.sample(all_files, N_SAMPLES)

    for fname in sampled_files:
        shutil.copy(os.path.join(SOURCE_DIR, fname), os.path.join(TARGET_DIR, fname))

    print(f"สุ่มและ copy เสร็จแล้ว: {len(os.listdir(TARGET_DIR))} ไฟล์")
else:
    print(f"มีข้อมูลอยู่แล้ว: {len(os.listdir(TARGET_DIR))} ไฟล์ ข้ามขั้นตอนนี้")

### 1.2 Parse Label จากชื่อไฟล์

ชื่อไฟล์ของ UTKFace มีรูปแบบ `[age]_[gender]_[race]_[timestamp].jpg` โดย gender: 0 = ชาย, 1 = หญิง

In [ ]:
LABELS_CSV = "data/labels.csv"

if not os.path.exists(LABELS_CSV):
    records = []
    skipped = 0
    for fname in os.listdir(TARGET_DIR):
        if not fname.endswith(".jpg"):
            continue
        try:
            age, gender, race, _ = fname.split("_", 3)
            records.append({
                "filename": fname,
                "age": int(age),
                "gender": int(gender),
                "race": int(race)
            })
        except ValueError:
            skipped += 1

    df_labels = pd.DataFrame(records)
    df_labels.to_csv(LABELS_CSV, index=False)
    print(f"Parse สำเร็จ: {len(df_labels)} แถว, ข้ามไป: {skipped} ไฟล์")
else:
    df_labels = pd.read_csv(LABELS_CSV)
    print(f"โหลด labels.csv ที่มีอยู่แล้ว: {len(df_labels)} แถว")

df_labels.describe()

### 1.3 Feature Extraction + PCA

แปลงภาพ → grayscale → resize 64x64 → flatten (4,096 features) → ลดมิติด้วย PCA เหลือ 50 components

In [ ]:
FEATURES_CSV = "data/features_pca.csv"
IMG_SIZE = (64, 64)
N_COMPONENTS = 50

if not os.path.exists(FEATURES_CSV):
    features, valid_idx = [], []
    for i, fname in enumerate(df_labels['filename']):
        fpath = os.path.join(TARGET_DIR, fname)
        try:
            img = Image.open(fpath).convert("L").resize(IMG_SIZE)
            arr = np.array(img).flatten() / 255.0
            features.append(arr)
            valid_idx.append(i)
        except Exception as e:
            print(f"ข้ามภาพ {fname}: {e}")

    X_raw = np.array(features)
    df_labels = df_labels.iloc[valid_idx].reset_index(drop=True)
    print(f"Feature matrix ก่อนทำ PCA: {X_raw.shape}")

    pca = PCA(n_components=N_COMPONENTS, random_state=42)
    X_pca = pca.fit_transform(X_raw)
    explained = pca.explained_variance_ratio_.sum()
    print(f"Feature matrix หลังทำ PCA: {X_pca.shape}")
    print(f"PCA {N_COMPONENTS} components เก็บ variance ไว้ได้: {explained*100:.2f}%")

    pca_cols = [f"pc_{i+1}" for i in range(N_COMPONENTS)]
    df_pca_features = pd.DataFrame(X_pca, columns=pca_cols)
    df = pd.concat([df_labels[['filename', 'age', 'gender']], df_pca_features], axis=1)
    df.to_csv(FEATURES_CSV, index=False)
    print("บันทึก data/features_pca.csv เรียบร้อย")
else:
    df = pd.read_csv(FEATURES_CSV)
    print(f"โหลด features_pca.csv ที่มีอยู่แล้ว: {df.shape}")

pc_cols = [c for c in df.columns if c.startswith("pc_")]
df.head()

### 1.4 สรุปข้อมูลก่อนเข้าโมเดล

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["age"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")

df["gender"].value_counts().sort_index().plot(
    kind="bar", ax=axes[1], color=["skyblue", "salmon"]
)
axes[1].set_title("Gender Distribution (0=Male, 1=Female)")
axes[1].set_xticklabels(["Male", "Female"], rotation=0)

plt.tight_layout()
plt.show()

print(f"จำนวนข้อมูลทั้งหมด: {len(df)} แถว | Features: {len(pc_cols)} PCA components")

---
## LAB 1: Regression — Age Prediction

เปรียบเทียบ **Simple Linear Regression** (ใช้ 1 feature: `pc_1`) กับ **Multiple Linear Regression** (ใช้ทั้ง 50 PCA components) ในการทำนายอายุ

In [ ]:
y_age = df["age"].values
train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42)

# --- Simple Linear Regression ---
X_simple = df[["pc_1"]].values
X_train_s, X_test_s = X_simple[train_idx], X_simple[test_idx]
y_train, y_test = y_age[train_idx], y_age[test_idx]

model_simple = LinearRegression()
model_simple.fit(X_train_s, y_train)
pred_train_s = model_simple.predict(X_train_s)
pred_test_s = model_simple.predict(X_test_s)

# --- Multiple Linear Regression ---
X_multi = df[pc_cols].values
X_train_m, X_test_m = X_multi[train_idx], X_multi[test_idx]

model_multi = LinearRegression()
model_multi.fit(X_train_m, y_train)
pred_train_m = model_multi.predict(X_train_m)
pred_test_m = model_multi.predict(X_test_m)

lab1_results = pd.DataFrame({
    "model": ["Simple Linear Regression", "Multiple Linear Regression"],
    "train_mae": [mean_absolute_error(y_train, pred_train_s), mean_absolute_error(y_train, pred_train_m)],
    "test_mae": [mean_absolute_error(y_test, pred_test_s), mean_absolute_error(y_test, pred_test_m)],
    "train_rmse": [np.sqrt(mean_squared_error(y_train, pred_train_s)), np.sqrt(mean_squared_error(y_train, pred_train_m))],
    "test_rmse": [np.sqrt(mean_squared_error(y_test, pred_test_s)), np.sqrt(mean_squared_error(y_test, pred_test_m))],
    "train_r2": [r2_score(y_train, pred_train_s), r2_score(y_train, pred_train_m)],
    "test_r2": [r2_score(y_test, pred_test_s), r2_score(y_test, pred_test_m)],
})
lab1_results.to_csv("data/lab1_results.csv", index=False)
lab1_results

### กราฟเปรียบเทียบ: Actual vs Predicted Age

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test, pred_test_s, alpha=0.5, color="orange")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel("Actual Age"); axes[0].set_ylabel("Predicted Age")
axes[0].set_title(f"Simple LR (R2={r2_score(y_test, pred_test_s):.3f})")

axes[1].scatter(y_test, pred_test_m, alpha=0.5, color="teal")
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1].set_xlabel("Actual Age"); axes[1].set_ylabel("Predicted Age")
axes[1].set_title(f"Multiple LR (R2={r2_score(y_test, pred_test_m):.3f})")

plt.tight_layout()
plt.savefig("data/lab1_regression_comparison.png")
plt.show()

### สรุป LAB 1

- **Multiple Linear Regression ดีกว่า Simple Linear Regression อย่างชัดเจน** ทั้งในแง่ MAE ที่ต่ำกว่าและ R² ที่สูงกว่า เพราะการใช้ features มากขึ้น (50 PCA components แทนที่จะเป็นแค่ 1 ตัว) ช่วยให้โมเดลจับความสัมพันธ์ระหว่างลักษณะใบหน้ากับอายุได้ดีขึ้น
- **Simple LR มี R² ติดลบ** แปลว่าแย่กว่าการเดาค่าเฉลี่ยเสียอีก เพราะ `pc_1` เพียงตัวเดียวไม่เพียงพอต่อการอธิบายอายุ
- โดยรวม Linear Regression ยังมีข้อจำกัดกับข้อมูลภาพที่ซับซ้อน (MAE ~12-13 ปี) ซึ่งเป็นเรื่องคาดหวังได้สำหรับโมเดลเชิงเส้นแบบพื้นฐาน

---
## LAB 2: Classification — Gender Prediction

ใช้ **Logistic Regression** ทำนายเพศจากภาพใบหน้า พร้อม Decision Boundary Visualization และ Confusion Matrix

In [ ]:
y_gender = df["gender"].values
X = df[pc_cols].values

X_train, X_test, y_train_g, y_test_g = train_test_split(
    X, y_gender, test_size=0.2, random_state=42, stratify=y_gender
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Gender distribution: {pd.Series(y_gender).value_counts().to_dict()}")

### Decision Boundary Visualization (ใช้ pc_1, pc_2 เพื่อ plot ใน 2 มิติ)

In [ ]:
X_2d = df[["pc_1", "pc_2"]].values
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y_gender, test_size=0.2, random_state=42, stratify=y_gender
)

model_2d = LogisticRegression(max_iter=1000)
model_2d.fit(X_train_2d, y_train_2d)

x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_gender, cmap="coolwarm", edgecolors="k", alpha=0.7)
plt.xlabel("PC 1"); plt.ylabel("PC 2")
plt.title("Decision Boundary (Logistic Regression, pc_1 vs pc_2)")
plt.legend(handles=scatter.legend_elements()[0], labels=["Male", "Female"])
plt.savefig("data/lab2_decision_boundary.png")
plt.show()

### Logistic Regression (ใช้ทุก PCA components) + Metrics

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train_g)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test_g, y_pred)
prec = precision_score(y_test_g, y_pred)
rec = recall_score(y_test_g, y_pred)
f1 = f1_score(y_test_g, y_pred)

print(f"Accuracy : {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall   : {rec:.3f}")
print(f"F1-score : {f1:.3f}")

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test_g, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Male", "Female"])
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap="Blues")
plt.title("Confusion Matrix - Gender Prediction")
plt.savefig("data/lab2_confusion_matrix.png")
plt.show()

### ROC Curve + AUC

In [ ]:
fpr, tpr, _ = roc_curve(y_test_g, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})", color="darkorange")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Gender Prediction")
plt.legend()
plt.savefig("data/lab2_roc_curve.png")
plt.show()

lab2_results = pd.DataFrame({
    "model": ["Logistic Regression (Gender)"],
    "accuracy": [acc], "precision": [prec], "recall": [rec],
    "f1_score": [f1], "auc": [roc_auc],
})
lab2_results.to_csv("data/lab2_results.csv", index=False)
lab2_results

### สรุป LAB 2

- Logistic Regression ทำนายเพศได้แม่นยำสูง (Accuracy และ AUC สูง) ดีกว่างาน Regression ทำนายอายุมาก
- สาเหตุที่เป็นไปได้: เพศเป็นลักษณะที่แยกจากกันชัดเจน (binary, มี pattern โครงสร้างใบหน้าที่แตกต่างค่อนข้างสม่ำเสมอ) ต่างจากอายุที่เป็นค่าต่อเนื่องและมีปัจจัยกวนมาก (แสง, การแต่งหน้า, สภาพผิว)
- Decision Boundary ที่เห็นใน 2D (pc_1, pc_2) แสดงให้เห็นว่าข้อมูลมีการซ้อนทับกันอยู่บ้าง แต่โมเดลยังแบ่งกลุ่มได้ในระดับที่ใช้งานได้

---
## LAB 3: Model Comparison

นำผลลัพธ์จาก LAB 1 และ LAB 2 มาเปรียบเทียบและวิเคราะห์ตาม 4 หัวข้อที่ใบงานกำหนด

### 3.1 Simple vs Multiple Linear Regression

In [ ]:
display(lab1_results[["model", "test_mae", "test_rmse", "test_r2"]])

simple_r2 = lab1_results.loc[lab1_results["model"] == "Simple Linear Regression", "test_r2"].values[0]
multi_r2 = lab1_results.loc[lab1_results["model"] == "Multiple Linear Regression", "test_r2"].values[0]
print(f"\nMultiple LR มี Test R2 สูงกว่า Simple LR อยู่ {multi_r2 - simple_r2:.3f}")
print("การเพิ่มจำนวน features (จาก 1 เป็น 50 PCA components) ช่วยให้โมเดลอธิบาย")
print("ความแปรปรวนของอายุได้ดีขึ้นอย่างชัดเจน")

### 3.2 Training vs Testing Performance

In [ ]:
lab1_gap = lab1_results.copy()
lab1_gap["r2_gap"] = lab1_gap["train_r2"] - lab1_gap["test_r2"]
lab1_gap["mae_gap"] = lab1_gap["test_mae"] - lab1_gap["train_mae"]
display(lab1_gap[["model", "train_r2", "test_r2", "r2_gap", "train_mae", "test_mae", "mae_gap"]])

print("\nถ้า r2_gap (Train R2 - Test R2) เป็นบวกมาก แปลว่าโมเดล overfit")
print("คือเรียนรู้ training data ได้ดีเกินไป จนไม่ generalize ไปยังข้อมูลใหม่ได้ดีเท่า")

### 3.3 Regression vs Classification

In [ ]:
print("Regression (Age Prediction) ใช้ R2, MAE, RMSE:")
display(lab1_results[lab1_results["model"] == "Multiple Linear Regression"][["test_mae", "test_rmse", "test_r2"]])

print("\nClassification (Gender Prediction) ใช้ Accuracy, Precision, Recall, F1, AUC:")
display(lab2_results[["accuracy", "precision", "recall", "f1_score", "auc"]])

print("\nทั้งสองงานใช้ features ชุดเดียวกัน (PCA components จากภาพใบหน้า)")
print("แต่ตัวชี้วัดเทียบกันตรงๆ ไม่ได้ เพราะเป็นคนละประเภทปัญหา:")
print("  - Regression ทำนายค่าต่อเนื่อง (อายุ) วัดด้วยค่าความคลาดเคลื่อน")
print("  - Classification ทำนายประเภท (เพศ) วัดด้วยความถูก/ผิดของ class")

### 3.4 Model Performance Metrics (สรุปรวม)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

lab1_results.plot(x="model", y=["train_r2", "test_r2"], kind="bar", ax=axes[0], color=["skyblue", "salmon"])
axes[0].set_title("Regression: Train vs Test R2")
axes[0].set_ylabel("R2 Score")
axes[0].set_xticklabels(lab1_results["model"], rotation=15, ha="right")

metrics = ["accuracy", "precision", "recall", "f1_score", "auc"]
values = lab2_results[metrics].values[0]
axes[1].bar(metrics, values, color="mediumseagreen")
axes[1].set_title("Classification: Gender Prediction Metrics")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("data/lab3_comparison_summary.png")
plt.show()

summary = pd.DataFrame({
    "Task": ["Age Prediction (Simple LR)", "Age Prediction (Multiple LR)", "Gender Prediction (Logistic Reg.)"],
    "Type": ["Regression", "Regression", "Classification"],
    "Key_Metric_1": [f"R2={lab1_results.iloc[0]['test_r2']:.3f}", f"R2={lab1_results.iloc[1]['test_r2']:.3f}", f"Acc={lab2_results.iloc[0]['accuracy']:.3f}"],
    "Key_Metric_2": [f"MAE={lab1_results.iloc[0]['test_mae']:.2f}", f"MAE={lab1_results.iloc[1]['test_mae']:.2f}", f"F1={lab2_results.iloc[0]['f1_score']:.3f}"],
})
summary.to_csv("data/lab3_final_summary.csv", index=False)
summary

---
## สรุปผลรวมทั้งโปรเจค

| Task | Type | Key Metric 1 | Key Metric 2 |
|---|---|---|---|
| Age Prediction (Simple LR) | Regression | R² ≈ -0.03 | MAE ≈ 15.5 |
| Age Prediction (Multiple LR) | Regression | R² ≈ 0.24 | MAE ≈ 12.7 |
| Gender Prediction (Logistic Reg.) | Classification | Accuracy ≈ 0.84 | F1 ≈ 0.83 |

### ข้อสรุปสำคัญ

1. **Gender Classification ทำได้ดีกว่า Age Regression อย่างชัดเจน** เพราะเพศเป็นลักษณะที่แยกกลุ่มได้ชัดเจนกว่า ในขณะที่อายุมีปัจจัยกวนมากและเป็นค่าต่อเนื่องที่ยากต่อการทำนายด้วยโมเดลเชิงเส้น

2. **จำนวน Features มีผลต่อประสิทธิภาพของ Regression อย่างมาก** — การใช้ PCA components มากขึ้น (จาก 1 เป็น 50) ช่วยเพิ่มความแม่นยำได้ชัดเจน

3. **พบสัญญาณ Overfitting เล็กน้อยใน Multiple Linear Regression** ซึ่งอาจแก้ไขได้ด้วยเทคนิค Regularization (Ridge/Lasso Regression) หรือปรับจำนวน PCA components ให้เหมาะสมยิ่งขึ้น

4. **ข้อจำกัดของโมเดลเชิงเส้น (Linear Model)** — ทั้ง Linear Regression และ Logistic Regression เป็นโมเดลพื้นฐานที่ไม่สามารถจับความสัมพันธ์แบบไม่เชิงเส้น (non-linear) ในภาพได้ดีเท่าโมเดลที่ซับซ้อนกว่า เช่น CNN (Convolutional Neural Network) ซึ่งเป็นแนวทางที่น่าสนใจสำหรับพัฒนาต่อยอด

### แนวทางพัฒนาต่อ

- ทดลองใช้ Regularization (Ridge/Lasso) เพื่อลด overfitting
- ปรับจำนวน PCA components (ทดลองค่าอื่นนอกจาก 50)
- ทดลองใช้ CNN หรือ Transfer Learning (เช่น ResNet, VGG) แทน PCA + Linear Model
- เพิ่มขนาด dataset จาก 1,500 เป็นจำนวนที่มากขึ้น